In [ ]:
import numpy as np

def convolve(input_tensor, filters, stride=(1, 1), padding='valid', dilation=(1, 1)):
    """
    Аналог conv2d, реализованный с использованием numpy.

    Параметры:
    - input_tensor: numpy.ndarray, форма - (C, H, W), формат - "channels-first", C - кол-во каналов, H - высота тензора, W - его ширина
    - filters: numpy.ndarray, формат - (M, C, R, S), M - кол-во фильтров
    - stride: двумерный tuple, шаг перемещения (<по x>, <по y>)
    - padding: string ("valid" или "same"), тип отступа (по аналогии с conv2D в keras или pytorch)
    - dilation: двумерный tuple, растяжение свертки (<по x>, <по y>)
    """

    C, H, W = input_tensor.shape
    M, C_f, R, S = filters.shape

    if C != C_f:
        raise ValueError("Количество каналов фильтров и входного тензора должно совпадать")

    if R > H or S > W:
        raise ValueError("Размеры фильтра превышают размеры входного тензора")

    if max(stride) > 1 and max(dilation) > 1:
        raise ValueError("stride > 1 несовместимы с dilation > 1 (взято из документации keras)")

    # Вычисление размера фильтров с учетом растяжения свертки
    dilated_R = (R - 1) * dilation[0] + 1
    dilated_S = (S - 1) * dilation[1] + 1

    # Вычисление выходных размеров
    if padding == 'same':
        pad_height = max((H - 1) * stride[0] + dilated_R - H, 0)
        pad_width = max((W - 1) * stride[1] + dilated_S - W, 0)
        pad_top = pad_height // 2
        pad_bottom = pad_height - pad_top
        pad_left = pad_width // 2
        pad_right = pad_width - pad_left
    elif padding == 'valid':
        pad_top = pad_bottom = pad_left = pad_right = 0
    else:
        raise ValueError("Недопустимый тип отступа")

    if padding == 'same' and (stride[0] > 1 or stride[1] > 1):
        raise ValueError("Использование типа отсупа \"Same\" возможно только со stride = (1, 1) (взято из документации pytorch)")

    # Дополнение входного тензора
    padded_input = np.pad(input_tensor, ((0, 0), (pad_top, pad_bottom), (pad_left, pad_right)), mode='constant', constant_values=0)

    # Высота и ширина выходного тензора
    OH = (H + pad_top + pad_bottom - dilated_R) // stride[0] + 1
    OW = (W + pad_left + pad_right - dilated_S) // stride[1] + 1

    # Инициализация выходного тензора
    output = np.zeros((M, OH, OW))

    # Выполнение свертки по формуле
    for m in range(M):
        for x in range(OH):
            for y in range(OW):
                for i in range(R):
                    for j in range(S):
                        for k in range(C):
                            x_in = x * stride[0] + i * dilation[0]
                            y_in = y * stride[1] + j * dilation[1]
                            if x_in < padded_input.shape[1] and y_in < padded_input.shape[2]:
                                output[m, x, y] += (padded_input[k, x_in, y_in] * filters[m, k, i, j])
    return output

np.random.seed(11)
input_tensor = np.random.rand(3, 20, 20)  # 3 канала, размеры тензора 20x20

np.random.seed(12)
filters = np.random.rand(2, 3, 5, 5)  # 2 фильтра, 3 канала, размер фильтра 5x5

output = convolve(input_tensor, filters, stride=(1, 1), padding='same', dilation=(1, 1))

print("Форма выходного тензора:", output.shape)
%timeit -n 100 convolve(input_tensor, filters, stride=(1, 1), padding='same', dilation=(1, 1))

Форма выходного тензора: (2, 20, 20)
88.1 ms ± 3.54 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
import numpy as np
from timeit import default_timer as timer

def depthwise_separable_convolution(input_tensor, depthwise_filters, pointwise_filters, stride=(1, 1), padding='valid', dilation=(1, 1)):
    """
    Параметры:
    - input_tensor: numpy.ndarray, форма - (C, H, W), формат - "channels-first", C - кол-во каналов, H - высота тензора, W - его ширина
    - depthwise_filters: numpy.ndarray, фильтры по глубине в формате "channels-first" - (C, R, S)
    - pointwise_filters: numpy.ndarray, поточечные фильтры в форме (M, C), M - кол-во выходных каналов, C - входных
    """

    C, H, W = input_tensor.shape
    C_d, R, S = depthwise_filters.shape
    M, C_p = pointwise_filters.shape

    if C != C_d or C != C_p:
        raise ValueError("Количество каналов во входном, поточечном и фильтре по глубине должны совпадать")

    if R > H or S > W:
        raise ValueError("Размеры фильтра превышают размеры входного тензора")

    if max(stride) > 1 and max(dilation) > 1:
        raise ValueError("stride > 1 несовместимы с dilation > 1 (взято из документации keras)")

    dilated_R = (R - 1) * dilation[0] + 1
    dilated_S = (S - 1) * dilation[1] + 1

    if padding == 'same':
        pad_height = max((H - 1) * stride[0] + dilated_R - H, 0)
        pad_width = max((W - 1) * stride[1] + dilated_S - W, 0)
        pad_top = pad_height // 2
        pad_bottom = pad_height - pad_top
        pad_left = pad_width // 2
        pad_right = pad_width - pad_left
    elif padding == 'valid':
        pad_top = pad_bottom = pad_left = pad_right = 0
    else:
        raise ValueError("Недопустимый тип отступа")

    if padding == 'same' and (stride[0] > 1 or stride[1] > 1):
        raise ValueError("Использование типа отсупа \"Same\" возможно только со stride = (1, 1) (взято из документации pytorch)")

    padded_input = np.pad(input_tensor, ((0, 0), (pad_top, pad_bottom), (pad_left, pad_right)), mode='constant', constant_values=0)

    OH = (H + pad_top + pad_bottom - dilated_R) // stride[0] + 1
    OW = (W + pad_left + pad_right - dilated_S) // stride[1] + 1

    # Свертка по глубине
    depthwise_output = np.zeros((C, OH, OW))
    for c in range(C):
        for x in range(OH):
            for y in range(OW):
                for i in range(R):
                    for j in range(S):
                        x_in = x * stride[0] + i * dilation[0]
                        y_in = y * stride[1] + j * dilation[1]
                        if x_in < padded_input.shape[1] and y_in < padded_input.shape[2]:
                            depthwise_output[c, x, y] += padded_input[c, x_in, y_in] * depthwise_filters[c, i, j]

    # Поточечная свертка
    output = np.zeros((M, OH, OW))
    for m in range(M):
        for x in range(OH):
            for y in range(OW):
                for c in range(C):
                    output[m, x, y] += depthwise_output[c, x, y] * pointwise_filters[m, c]

    return output

np.random.seed(11)
input_tensor = np.random.rand(3, 20, 20)

np.random.seed(12)
depthwise_filters = np.random.rand(3, 5, 5)

np.random.seed(13)
pointwise_filters = np.random.rand(2, 3) # 2 выходных канала, 3 входных

output = depthwise_separable_convolution(input_tensor, depthwise_filters, pointwise_filters, stride=(1, 1), padding='same', dilation=(1, 1))

print("Форма выходного тензора:", output.shape)
%timeit -n 100 depthwise_separable_convolution(input_tensor, depthwise_filters, pointwise_filters, stride=(1, 1), padding='same', dilation=(1, 1))

Форма выходного тензора: (2, 20, 20)
45.7 ms ± 6.82 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


**Видим, что время исполнения функции convolve() почти в два раза превышает время исполнения функции depthwise_separable_convolution(). Это происходит из-за сокращения количества выполняемых вычислений в depthwise_separable_convolution() [https://eli.thegreenplace.net/2018/depthwise-separable-convolutions-for-machine-learning/].**